In [ ]:
import sys
from pathlib import Path

import pandas as pd
from pandas.testing import assert_frame_equal
from prefect.logging import disable_run_logger

CURRENT_FILE = Path(__file__).resolve()
PROJECT_ROOT = CURRENT_FILE.parent.parent  # job assignment
PART2_PATH = PROJECT_ROOT / "Part 2 Data Cleaning & Pipeline Orchestration (Python + Prefect)"

sys.path.insert(0, str(PART2_PATH))

from pipeline import transform_customers, transform_orders


def test_transform_customers_with_dummy_data():
    input_df = pd.DataFrame(
        {
            "customer_id": [1, 1, 2, 3, 3, 4, 5, 6],
            "name": ["Alice","Alice","Bob","Charlie","Charlie","Diana","Evan","Fiona",],
            "email": [None,"alice@example.com",None,"charlie@old.com",None,"diana@example.com",
                      None,"fiona@example.com",],
            "phone": ["(040) 123-4567","0401 999 888","+61 412-345-678","03-9999-8888","(03) 7777 6666",
                      None,"abc-0400-555-111","",],
            "signup_date": ["2024-01-10","2024-02-15","2024-03-01","2024-01-05","2024-04-20","2024-02-11",
                            "2024-05-01","2024-06-01",],
        }
    )
    
    expected = pd.DataFrame(
        {
            "customer_id": [1, 2, 3, 4, 5, 6],
            "name": ["Alice","Bob","Charlie","Diana","Evan","Fiona",],
            "email": [
                "alice@example.com",
                "unknown@domain.com",
                "unknown@domain.com",
                "diana@example.com",
                "unknown@domain.com",
                "fiona@example.com",
            ],
            "phone": [
                "0401999888",
                "61412345678",
                "0377776666",
                None,
                "0400555111",
                None,
            ],
            "signup_date": [
                pd.to_datetime("2024-02-15").date(),
                pd.to_datetime("2024-03-01").date(),
                pd.to_datetime("2024-04-20").date(),
                pd.to_datetime("2024-02-11").date(),
                pd.to_datetime("2024-05-01").date(),
                pd.to_datetime("2024-06-01").date(),
            ],
        }
    )

    with disable_run_logger():
        result = transform_customers.fn(input_df).reset_index(drop=True)

    result = result.where(pd.notna(result), None)
    assert_frame_equal(result, expected)